# 06 - Data loader
- **Step 1 (PI2):** load the listeners' hearing-aid (compressor) settings
- **Step 2 (PI3 + PI4):** a documented, reusable loader that returns mixture, stems, audiogram, gains and compressor settings for any scene, plus checks on every valid scene
- **Step 3 (PI5):** find the real recordings and compare them with the synthetic ones

**Before this, run `00_shared_drive_data.ipynb`:** the Drive cell, the clone-Clarity cell and **Part A**. Run **Part B** too if you want the eval set (the only real recordings we have, see Step 3).

Note: this notebook uses paths in `/content`, so a Colab session restart means running 00 again.

1. Check prerequisites (nothing is downloaded here)

In [ ]:
import os
from pathlib import Path
os.chdir('/content')

root = Path('/content/cadenza_data/cad2/task2')
clarity_dir = Path('/content/clarity')
meta = root / 'metadata'

need = {
    'Clarity (run 00 clone cell or 01_setup)': clarity_dir / 'recipes/cad2/task2/baseline/config.yaml',
    'valid metadata (run 00 Part A)': meta / 'scenes.valid.json',
    'valid audio (run 00 Part A)': root / 'audio/valid',
}
missing = [k for k, p in need.items() if not p.exists()]
assert not missing, 'Missing: ' + '; '.join(missing)

splits = [sp for sp in ['train', 'valid', 'eval'] if (meta / f'scenes.{sp}.json').exists()]
print('prerequisites ok | splits with metadata:', splits)
print('audio folders:', sorted(p.name for p in (root / 'audio').iterdir()))

2. Get the loader from the repo
The loader lives in `cadenza/cad2_loader.py` in our GitHub repo. This cell clones the repo (or pulls the latest version if it is already cloned) and imports it.
**To change the loader, edit `cadenza/cad2_loader.py`, push it, then re-run this cell.** Colab only sees what is pushed to the `new_model_creation` branch.

In [ ]:
!git clone -q -b new_model_creation https://github.com/stevejo12/hearing-adaptive-music-processor.git /content/project 2>/dev/null || git -C /content/project pull -q

import sys, json
sys.path.insert(0, '/content/project/cadenza')
from cad2_loader import CAD2Loader, mean_hearing_loss, clip_stats, stem_bleed

3. Create the loader

In [ ]:
loader = CAD2Loader(root, clarity_dir)
print('loader ready:', loader.root)

## Step 1 - Compressor settings (PI2)
Every listener has their own hearing-aid settings in `compressor_params.<split>.json`:
- `cr_l`, `cr_r` - **compression ratio** per ear (how much loud sounds are squashed)
- `gain_l`, `gain_r` - **makeup gain** per ear (how much everything is turned up)

The rest is the same for everyone and comes from the baseline `config.yaml`: crossover frequencies (band edges), attack, release and threshold per band.

1a. Look at the raw file

In [ ]:
comp = loader.compressor_params('valid')
first_id = next(iter(comp))
print(len(comp), 'listeners in compressor_params.valid.json')
print(first_id, comp[first_id])

1b. Combine with the shared settings - the same structure `enhance.py` builds (`mbc_params_listener`)

In [ ]:
import json
print('shared:', loader.shared_compressor_settings())
print(json.dumps(loader.get_compressor(first_id, 'valid'), indent=1))

1c. Every listener should have both an audiogram and compressor settings

In [ ]:
for sp in splits:
    try:
        print(sp, loader.check_listener_match(sp))
    except FileNotFoundError as e:
        print(sp, 'skipped:', e)

1d. Does worse hearing get more gain / more compression?
One point per valid listener. x = average audiogram level over both ears (dB HL, higher = worse hearing).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

listeners = loader.listeners('valid')
ids = [l for l in comp if l in listeners]
loss = [mean_hearing_loss(listeners[l]) for l in ids]
gain = [np.mean([comp[l]['gain_l'], comp[l]['gain_r']]) for l in ids]
ratio = [np.mean([comp[l]['cr_l'], comp[l]['cr_r']]) for l in ids]

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(loss, gain); ax[0].set(xlabel='mean hearing loss (dB HL)', ylabel='mean makeup gain (dB)', title='Gain vs hearing loss')
ax[1].scatter(loss, ratio); ax[1].set(xlabel='mean hearing loss (dB HL)', ylabel='mean compression ratio', title='Compression vs hearing loss')
plt.tight_layout(); plt.show()
print(f'correlation  loss-gain: {np.corrcoef(loss, gain)[0, 1]:.2f}   loss-ratio: {np.corrcoef(loss, ratio)[0, 1]:.2f}')

**Step 1 notes (fill in):**
- Keys per listener: ___ (per band or one value?)
- Listeners missing settings: ___
- Worse hearing -> more gain? ___ (correlation ___). More compression? ___

## Step 2 - Reusable data loader (PI3 + PI4)

How the files link together:

| From | File | Gives |
|---|---|---|
| scene_id | `scene_listeners.<split>.json` | which listeners hear this scene |
| scene_id | `scenes.<split>.json` | song name + gain id |
| song name | `music.<split>.json` | for mixture and each source: instrument, track path, start, duration |
| gain id | `gains.json` | dB change per source |
| listener_id | `listeners.<split>.json` | audiogram (left / right) |
| listener_id | `compressor_params.<split>.json` | hearing-aid settings (Step 1) |

`load_scene()` follows these links and returns everything in one dict. **Consistent format (PI3):** all audio is float32, stereo `(samples, 2)`, 44.1 kHz (or `target_sr`), same length, cut to the scene segment.

2a. Load one scene (same as notebooks 03 and 05)

In [ ]:
s = loader.load_scene('S50001', split='valid')
for k, v in s.items():
    if isinstance(v, np.ndarray):
        print(f'{k:13s} array {v.shape} {v.dtype}')
    elif k == 'stems':
        print(f'{k:13s}', {kk: vv.shape for kk, vv in v.items()})
    elif k == 'listener':
        print(f'{k:13s} Listener {v.id}')
    elif k == 'compressor':
        print(f'{k:13s} left ratio {v["left"]["ratio"]}, left gain {v["left"]["makeup_gain"]}')
    else:
        print(f'{k:13s}', v)

2b. Look and listen: the listener's audiogram and the mixture

In [ ]:
from IPython.display import Audio, display

lst = s['listener']
plt.figure(figsize=(6, 3.5))
plt.semilogx(lst.audiogram_left.frequencies, lst.audiogram_left.levels, 'o-', label='left')
plt.semilogx(lst.audiogram_right.frequencies, lst.audiogram_right.levels, 's-', label='right')
plt.gca().invert_yaxis(); plt.xlabel('frequency (Hz)'); plt.ylabel('hearing level (dB HL)')
plt.title(f'Audiogram {lst.id}'); plt.legend(); plt.grid(alpha=.3); plt.show()
display(Audio(s['mixture'].T, rate=s['sample_rate']))

2c. Cross-check against the baseline
Repeat the exact lines `enhance.py` uses to cut the mixture and build the compressor settings, and compare with the loader.

In [ ]:
import sys as _sys
_sys.path.insert(0, str(clarity_dir))
from clarity.utils.flac_encoder import read_flac_signal

songs = loader.music('valid'); scene = loader.scenes('valid')['S50001']
song_name = scene['music']; listener_id = s['listener_id']

# enhance.py: read + cut the mixture
mixture_signal, sr = read_flac_signal(filename=root / 'audio' / songs[song_name]['mixture']['track'])
start = songs[song_name]['mixture']['start']; end = start + songs[song_name]['mixture']['duration']
mixture_signal = mixture_signal[int(start * sr): int(end * sr), :]
n = min(len(mixture_signal), len(s['mixture']))
print('mixture same as enhance.py :', np.allclose(mixture_signal[:n], s['mixture'][:n], atol=1e-6),
    f'(lengths {len(mixture_signal)} vs {len(s["mixture"])})')

# enhance.py: compressor params
import yaml
cfg = yaml.safe_load(open(clarity_dir / 'recipes/cad2/task2/baseline/config.yaml'))['enhancer']
p = loader.compressor_params('valid')[listener_id]
mbc = {'left': {}, 'right': {}}
for ear in ['left', 'right']:
    mbc[ear]['release'] = cfg['release']; mbc[ear]['attack'] = cfg['attack']; mbc[ear]['threshold'] = cfg['threshold']
mbc['left']['ratio'] = p['cr_l']; mbc['right']['ratio'] = p['cr_r']
mbc['left']['makeup_gain'] = p['gain_l']; mbc['right']['makeup_gain'] = p['gain_r']
mine = loader.get_compressor(listener_id, 'valid')
print('compressor same as enhance.py:', all(mbc[e] == mine[e] for e in ['left', 'right']))

2d. Check every valid scene
For each scene: same shape for all audio, gains cover every source, and stems add up to the mixture (mismatch below -30 dB). Takes a few minutes; set `LIMIT` to a number for a quick run.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

LIMIT = None    # e.g. 20 for a quick test
scene_ids = list(loader.scenes('valid'))[:LIMIT]
check_df = pd.DataFrame([loader.check_scene(sc, 'valid') for sc in tqdm(scene_ids)])

print(f"{check_df['ok'].sum()} passed / {(~check_df['ok']).sum()} failed  (out of {len(check_df)} valid scenes)")
print('mixture mismatch (dB): median', round(check_df['mix_mismatch_db'].median(), 1), '| worst', round(check_df['mix_mismatch_db'].max(), 1))
display(check_df[~check_df['ok']].head(20))

2e. Same checks on a sample of train scenes (train files are full songs, so this also tests the cutting)

In [ ]:
if 'train' in splits:
    rng = np.random.default_rng(0)
    train_ids = list(loader.scenes('train'))
    sample = [train_ids[i] for i in rng.choice(len(train_ids), size=min(30, len(train_ids)), replace=False)]
    train_check_df = pd.DataFrame([loader.check_scene(sc, 'train') for sc in tqdm(sample)])
    print(f"{train_check_df['ok'].sum()} passed / {(~train_check_df['ok']).sum()} failed  (sample of {len(sample)} train scenes)")
    display(train_check_df[~train_check_df['ok']].head(20))
else:
    print('no train metadata - skipped')

**Step 2 notes (fill in):**
- Valid scenes checked: ___ passed / ___ failed. Typical mixture mismatch: ___ dB
- Same as `enhance.py` (mixture cut, compressor): ___
- Train sample: ___ passed / ___ failed
- Anything odd: ___

## Step 3 - Real recordings vs synthetic (PI5)

**What we know so far:** the training and validation data are **synthetic** (EnsembleSet, CadenzaWoodwind: rendered from scores). Clarity's dataset script mentions `URMP.zip` and `BACH10.zip` (**real** recordings) but notebook 00 does not unpack them; they are not in the shared Drive folder. The **eval** set (00 Part B) has paths like `eval/URMP/...`, so it is the only real data we have, and it may contain mixtures only.

3a. Where does each split's audio come from?

In [ ]:
from collections import Counter

rows = []
for sp in splits:
    for song_name in loader.music(sp):
        rows.append({'split': sp, 'song': song_name, 'dataset': loader.song_dataset(sp, song_name)})
origin_df = pd.DataFrame(rows)
display(origin_df.groupby(['split', 'dataset']).size().rename('songs').reset_index())

# first folders of the track paths, to see the raw layout
for sp in splits:
    prefixes = Counter('/'.join(v['track'].split('/')[:2]) for song in loader.music(sp).values() for v in song.values())
    print(sp, prefixes.most_common(6))

3b. Are the real-recording datasets anywhere on this machine? Do the eval stems exist?

In [ ]:
for base in [Path('/content/zenodo_download'), root / 'audio']:
    if base.exists():
        hits = [p for p in base.rglob('*') if p.is_dir() and any(n in p.name for n in ['URMP', 'BACH10', 'Bach10'])]
        print(base, '->', [str(h) for h in hits[:5]] or 'none')

if 'eval' in splits:
    ev = loader.load_scene(next(iter(loader.scenes('eval'))), split='eval')
    print('eval example:', ev['scene_id'], ev['dataset'], '| stems found:', list(ev['stems']), '| missing:', ev['missing_stems'])

**Also read** the Cadenza CAD2 Task 2 data page (https://cadenzachallenge.org/docs/cadenza2/Rebalancing%20Classical/rebalancing) and note what it calls the real-recording / tuning set, which split it is, and whether it needs a separate download.

3c. Statistics on a sample of scenes per group
Mixture: length, number of sources, loudness (LUFS), peak, crest factor (peak / RMS, how "spiky" the sound is), spectral centroid (how bright). Stems (when they exist): centroid per instrument and **bleed** (how much a stem looks like the other stems).

In [ ]:
N_PER_GROUP = 20
rng = np.random.default_rng(0)

scene_rows, stem_rows = [], []
for sp in splits:
    scenes_sp = loader.scenes(sp)
    by_dataset = {}
    for sc, info in scenes_sp.items():
        by_dataset.setdefault(loader.song_dataset(sp, info['music']), []).append(sc)
    for dataset, ids in by_dataset.items():
        pick = [ids[i] for i in rng.choice(len(ids), size=min(N_PER_GROUP, len(ids)), replace=False)]
        for sc in tqdm(pick, desc=f'{sp}/{dataset}'):
            try:
                s3 = loader.load_scene(sc, split=sp)
            except Exception as e:
                print('skip', sp, sc, type(e).__name__, e); continue
            row = {'split': sp, 'dataset': dataset, 'origin': s3['origin'], 'scene': sc, 'seconds': len(s3['mixture']) / s3['sample_rate'], 'n_sources': len(s3['instruments']), **clip_stats(s3['mixture'], s3['sample_rate'])}
            if len(s3['stems']) > 1 and not s3['missing_stems']:
                bleed = stem_bleed(s3['stems'], s3['sample_rate'])
                row['bleed_mean'] = float(np.nanmean(list(bleed.values())))
                for k, st in s3['stems'].items():
                    stem_rows.append({'split': sp, 'origin': s3['origin'], 'instrument': s3['instruments'][k],'bleed': bleed[k], **clip_stats(st, s3['sample_rate'])})
            scene_rows.append(row)

stats_df = pd.DataFrame(scene_rows)
if 'bleed_mean' not in stats_df:
    stats_df['bleed_mean'] = np.nan # no group had complete stems
stem_df = pd.DataFrame(stem_rows)
summary = stats_df.groupby(['origin', 'split', 'dataset']).agg(
    scenes=('scene', 'count'), seconds=('seconds', 'mean'), n_sources=('n_sources', 'mean'),
    lufs=('lufs', 'mean'), peak_db=('peak_db', 'mean'), crest_db=('crest_db', 'mean'),
    centroid_hz=('centroid_hz', 'mean'), bleed=('bleed_mean', 'mean')).round(2)
display(summary)

3d. Plots: real vs synthetic

In [ ]:
metrics = [m for m in ['lufs', 'crest_db', 'centroid_hz', 'bleed_mean'] if m in stats_df and stats_df[m].notna().any()]
fig, axes = plt.subplots(1, len(metrics), figsize=(4 * len(metrics), 3.5), squeeze=False)
for ax, m in zip(axes[0], metrics):
    groups = [(o, g[m].dropna()) for o, g in stats_df.groupby('origin') if g[m].notna().any()]
    ax.boxplot([g for _, g in groups]); ax.set_xticks(range(1, len(groups) + 1), [o for o, _ in groups]); ax.set_title(m)
plt.tight_layout(); plt.show()

if len(stem_df):
    display(stem_df.groupby(['instrument', 'origin'])[['centroid_hz', 'lufs', 'bleed']].mean().round(1).unstack('origin'))

3e. Save the tables to Drive

In [ ]:
out = Path('/content/drive/MyDrive/Cadenza Challenge/results/06_data_loader')
if not Path('/content/drive/MyDrive').exists():
    out = Path('/content/results_06'); print('Drive not mounted - saving locally only')
out.mkdir(parents=True, exist_ok=True)
check_df.to_csv(out / 'valid_checks.csv', index=False)
if 'train_check_df' in globals(): train_check_df.to_csv(out / 'train_checks.csv', index=False)
origin_df.to_csv(out / 'song_origins.csv', index=False)
stats_df.to_csv(out / 'scene_stats.csv', index=False)
stem_df.to_csv(out / 'stem_stats.csv', index=False)
print('saved to', out, sorted(p.name for p in out.iterdir()))

**Step 3 notes (fill in):**
- Datasets per split: ___
- Real recordings available? ___ (eval only? stems or mixtures only?) What the Cadenza page calls the tuning set: ___
- Biggest real vs synthetic differences (loudness / crest / brightness / bleed): ___
- Why that could hurt the pretrained separators: ___
- Note: URMP players were recorded separately, so bleed may be low even for real data - report what the numbers show.